<a href="https://colab.research.google.com/github/stylekkm051049-bit/Project/blob/%E0%B8%9E%E0%B8%B5%E0%B8%A3%E0%B8%9E%E0%B8%B1%E0%B8%92%E0%B8%99%E0%B9%8C-%E0%B8%A4%E0%B8%97%E0%B8%98%E0%B8%B4%E0%B9%8C%E0%B8%AA%E0%B8%A2%E0%B8%B2%E0%B8%A1/%E0%B8%AA%E0%B9%88%E0%B8%A7%E0%B8%99%E0%B8%97%E0%B8%B5%E0%B9%88%208-9%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ขั้นที่ 8 — วิเคราะห์ข้อมูลด้วย pandas และ SQL
## วิเคราะห์ด้วย PANDAS

In [ ]:

# คอร์สไหนมีผู้สมัครมากที่สุด
q1_pandas = (
    registration_df[
        registration_df["สถานะการลงทะเบียน"] == "ลงทะเบียนสำเร็จ"
    ]
    .groupby(["รหัสคอร์ส", "ชื่อคอร์ส"])
    .size()
    .reset_index(name="จำนวนผู้สมัคร")
    .sort_values("จำนวนผู้สมัคร", ascending=False)
)

display(q1_pandas)

In [ ]:

# คอร์สไหนรายได้มากที่สุด
q2_pandas = (
    registration_df[
        registration_df["สถานะการลงทะเบียน"] == "ลงทะเบียนสำเร็จ"
    ]
    .groupby(["รหัสคอร์ส", "ชื่อคอร์ส"])
    .agg(
        รายได้รวม=("ยอดชำระสุทธิ (บาท)", "sum"),
        จำนวนผู้สมัคร=("รหัสนักเรียน", "count"),
        ค่าเรียนเฉลี่ย=("ยอดชำระสุทธิ (บาท)", "mean")
    )
    .reset_index()
    .sort_values("รายได้รวม", ascending=False)
)

display(q2_pandas)


In [ ]:

# คอร์สไหนรายได้มากที่สุด
q2_pandas = (
    registration_df[
        registration_df["สถานะการลงทะเบียน"] == "ลงทะเบียนสำเร็จ"
    ]
    .groupby(["รหัสคอร์ส", "ชื่อคอร์ส"])
    .agg(
        รายได้รวม=("ยอดชำระสุทธิ (บาท)", "sum"),
        จำนวนผู้สมัคร=("รหัสนักเรียน", "count"),
        ค่าเรียนเฉลี่ย=("ยอดชำระสุทธิ (บาท)", "mean")
    )
    .reset_index()
    .sort_values("รายได้รวม", ascending=False)
)

display(q2_pandas)

In [ ]:

# โรงเรียนไหนมีนักเรียนลงทะเบียนมากที่สุด?
q4_pandas = (
    registration_df[
        registration_df["สถานะการลงทะเบียน"] == "ลงทะเบียนสำเร็จ"
    ]
    .groupby("โรงเรียน")
    .size()
    .reset_index(name="จำนวนผู้ลงทะเบียน")
    .sort_values("จำนวนผู้ลงทะเบียน", ascending=False)
)

display(q4_pandas)

In [ ]:

# รายได้รวมทั้งหมด
total_revenue = registration_df.loc[
    registration_df["สถานะการลงทะเบียน"] == "ลงทะเบียนสำเร็จ",
    "ยอดชำระสุทธิ (บาท)"
].sum()

print(
    "รายได้รวมทั้งหมด:",
    format_currency(total_revenue)
)

# วิเคราะห์คำถามเดียวกันด้วย SQL

In [ ]:
conn = sqlite3.connect("tutoring_school.db")

In [ ]:
# คอร์สไหนมีผู้สมัครมากที่สุด?
q1_sql = pd.read_sql_query("""
SELECT
    "รหัสคอร์ส",
    "ชื่อคอร์ส",
    COUNT(*) AS จำนวนผู้สมัคร
FROM registration
WHERE "สถานะการลงทะเบียน" = 'ลงทะเบียนสำเร็จ'
GROUP BY "รหัสคอร์ส", "ชื่อคอร์ส"
ORDER BY จำนวนผู้สมัคร DESC
""", conn)

display(q1_sql)

In [ ]:
# คอร์สไหนสร้างรายได้มากที่สุด?
q2_sql = pd.read_sql_query("""
SELECT
    "รหัสคอร์ส",
    "ชื่อคอร์ส",
    ROUND(SUM("ยอดชำระสุทธิ (บาท)"), 2) AS รายได้รวม
FROM registration
WHERE "สถานะการลงทะเบียน" = 'ลงทะเบียนสำเร็จ'
GROUP BY "รหัสคอร์ส", "ชื่อคอร์ส"
ORDER BY รายได้รวม DESC
""", conn)

display(q2_sql)

In [ ]:
# โรงเรียนไหนมีนักเรียนลงทะเบียนมากที่สุด?
q4_sql = pd.read_sql_query("""
SELECT
    "โรงเรียน",
    COUNT(*) AS จำนวนผู้ลงทะเบียน
FROM registration
WHERE "สถานะการลงทะเบียน" = 'ลงทะเบียนสำเร็จ'
GROUP BY "โรงเรียน"
ORDER BY จำนวนผู้ลงทะเบียน DESC
""", conn)

display(q4_sql)

In [ ]:
sql_total = pd.read_sql_query("""
SELECT
    ROUND(SUM("ยอดชำระสุทธิ (บาท)"), 2) AS รายได้รวมทั้งหมด
FROM registration
WHERE "สถานะการลงทะเบียน" = 'ลงทะเบียนสำเร็จ'
""", conn)

display(sql_total)

# ขั้นที่ 9 — สร้างกราฟและสรุปผล

In [ ]:
# ติดตั้งฟอนต์ไทย
!apt-get install -y fonts-thai-tlwg -qq

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import glob
import os

# ค้นหาไฟล์ฟอนต์ไทยทั้งหมด
thai_fonts = glob.glob(
    "/usr/share/fonts/**/*.ttf",
    recursive=True
)

# แสดงฟอนต์ที่พบ
thai_fonts = [
    f for f in thai_fonts
    if any(x in os.path.basename(f).lower()
           for x in ["garuda", "loma", "waree", "kinnari", "tlwg"])
]

print("ฟอนต์ไทยที่พบ:")
for f in thai_fonts:
    print(f)

# ใช้ฟอนต์ตัวแรก
font_path = thai_fonts[0]

# เพิ่มฟอนต์เข้า Matplotlib
fm.fontManager.addfont(font_path)

# สร้าง FontProperties จากไฟล์โดยตรง
thai_font = fm.FontProperties(fname=font_path)

print("กำลังใช้:", thai_font.get_name())
print("ไฟล์:", font_path)

In [ ]:
# กราฟ 1: จำนวนผู้สมัครแต่ละคอร์ส
# Bar Chart

plt.figure(figsize=(10, 6))
plt.bar( q1_pandas["ชื่อคอร์ส"], q1_pandas["จำนวนผู้สมัคร"])
plt.title( "จำนวนผู้สมัครแยกตามคอร์ส", fontproperties=thai_font)
plt.xlabel( "คอร์ส", fontproperties=thai_font)
plt.ylabel("จำนวนผู้สมัคร (คน)", fontproperties=thai_font)
plt.xticks( rotation=45, ha="right", fontproperties=thai_font)
plt.yticks(fontproperties=thai_font)

plt.tight_layout()
plt.show()

# สรุปคอร์สที่มีผู้สมัครมากที่สุด
max_applicants = q1_pandas["จำนวนผู้สมัคร"].max()
top_courses = q1_pandas[ q1_pandas["จำนวนผู้สมัคร"] == max_applicants]
print("คอร์สที่มีผู้สมัครมากที่สุด จำนวน", max_applicants, "คน ได้แก่")
for _, row in top_courses.iterrows():
    print("-", row["ชื่อคอร์ส"])


In [ ]:
# กราฟ 2: รายได้ของแต่ละคอร์ส
# Line Chart
plt.figure(figsize=(10, 5))
plt.plot( q2_pandas["ชื่อคอร์ส"], q2_pandas["รายได้รวม"], marker="o" )
plt.title( "รายได้รวมแยกตามคอร์ส", fontproperties=thai_font )
plt.xlabel( "คอร์ส", fontproperties=thai_font )
plt.ylabel( "รายได้รวม (บาท)", fontproperties=thai_font )
plt.xticks( rotation=45, ha="right", fontproperties=thai_font )
plt.yticks(fontproperties=thai_font)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

top_revenue = q2_pandas.iloc[0]
print( "สรุปผล: คอร์สที่สร้างรายได้สูงที่สุดคือ", top_revenue["ชื่อคอร์ส"], "ด้วยรายได้รวม", f"{top_revenue['รายได้รวม']:,.2f}", "บาท" )

In [ ]:
# โรงเรียนไหนมีผู้ลงทะเบียนมากที่สุด
top_school = q4_pandas.loc[ q4_pandas["จำนวนผู้ลงทะเบียน"].idxmax() ]

plt.figure(figsize=(10, 6))
plt.bar( q4_pandas["โรงเรียน"], q4_pandas["จำนวนผู้ลงทะเบียน"] )
plt.title( "จำนวนผู้ลงทะเบียนแยกตามโรงเรียน", fontproperties=thai_font )
plt.xlabel( "โรงเรียน", fontproperties=thai_font )
plt.ylabel( "จำนวนผู้ลงทะเบียน (คน)", fontproperties=thai_font )
plt.xticks( rotation=45, ha="right", fontproperties=thai_font )
plt.yticks(fontproperties=thai_font)

plt.tight_layout()
plt.show()


print( "สรุปผล: โรงเรียนที่มีนักเรียนลงทะเบียนมากที่สุดคือ", top_school["โรงเรียน"], "จำนวน", top_school["จำนวนผู้ลงทะเบียน"], "คน" )

# จากการวิเคราะห์ข้อมูลพบว่า โรงเรียนแก่นนครวิทยาลัย มีนักเรียนลงทะเบียนมากที่สุด จำนวน 52 คน แสดงให้เห็นว่าเป็นกลุ่มเป้าหมายที่มีความสนใจในการเรียนสูง คอร์สที่มีผู้สมัครมากที่สุดมีทั้งหมด 8 คอร์ส โดยมีผู้สมัครเท่ากันที่ 40 คน ได้แก่ คณิตศาสตร์ ม.ปลาย, ฟิสิกส์ ม.ปลาย, เคมี ม.ปลาย, ชีววิทยา ม.ปลาย, คณิตศาสตร์ A-Level, ฟิสิกส์ A-Level, ภาษาอังกฤษ A-Level และเคมี A-Level นอกจากนี้ คณิตศาสตร์ A-Level เป็นคอร์สที่สร้างรายได้สูงที่สุด โดยมีรายได้รวม 132,825 บาท ดังนั้นโรงเรียนควรพิจารณาเพิ่มรอบเรียนและจัดสรรที่นั่งสำหรับคอร์สที่ได้รับความนิยม รวมถึงใช้ข้อมูลของโรงเรียนและคอร์สที่มีความต้องการสูงในการวางแผนการตลาดต่อไปครับ